# 01 Grid: H3 cells for the Helsinki, Espoo and Sipoo sea areas

Phase 0 spike. The question: can we build the scoring grid from spec §"Spatial model"?

- **res 10** (~65 m edge, ~0.015 km²) for water within ~200 m of the shore
- **res 9** (~174 m edge, ~0.1 km²) for water further out
- land and inland lakes removed

Spec estimate: "a few thousand to ~30k cells". This notebook gives the real number.

**Data:** Traficom chart depth areas (`DepthArea_A`), CC BY 4.0, source: Traficom. One live WFS request, ~3k polygons, ~20 s.

In [ ]:
import geopandas as gpd
import h3
from shapely.geometry import Polygon, shape
import folium

## 1. Domain polygon

A rough hand-drawn outline in lon/lat (EPSG:4326). It runs from the Espoo/Kirkkonummi border in the west (~24.5°E) to the Sipoo/Porvoo border in the east (~25.62°E), and from the inner bays in the north out past the outer archipelago in the south.

It doesn't need to follow the coast. The northern edge sits on land on purpose, because land gets cut away in step 2. Later this polygon moves into the region config (spec: "Regions as configuration").

All distance and area work below uses **EPSG:3067** (ETRS-TM35FIN, metres), the Finnish national projection. Traficom also delivers data in this CRS.

In [ ]:
domain = gpd.GeoSeries(
    [Polygon([(24.50, 60.20), (24.50, 60.05), (24.95, 59.95), (25.62, 60.02), (25.62, 60.32), (24.50, 60.32)])],
    crs=4326,
).to_crs(3067)
domain_poly = domain.iloc[0]
print(f"domain area: {domain_poly.area / 1e6:.0f} km²")

## 2a. Water from Traficom depth areas

`DepthArea_A` polygons cover the charted sea, each with a depth range (`DRVAL1`–`DRVAL2`, m). We only need their footprint: **the union of all depth areas = water**. Everything else in the domain is land.

This also drops inland lakes for free, because sea charts don't cover them.

The server returns everything in one request when `count` is high enough. The assert catches the day it starts truncating.

In [ ]:
MAX = 10_000
minx, miny, maxx, maxy = domain.total_bounds
url = (
    "https://julkinen.traficom.fi/inspirepalvelu/rajoitettu/wfs?service=WFS&version=2.0.0"
    "&request=GetFeature&typeNames=rajoitettu:DepthArea_A&outputFormat=application/json"
    f"&bbox={minx},{miny},{maxx},{maxy},EPSG:3067&count={MAX}"
)
depth = gpd.read_file(url)
assert len(depth) < MAX, "WFS hit the count limit; add paging"
print(len(depth), "depth areas,", depth.crs)
depth[["DRVAL1", "DRVAL2", "SORDAT"]].head()

## 2b. Split water into nearshore and offshore

- `water` = union of depth areas, clipped to the domain
- `land` = domain minus water (mainland and islands)
- `near` = water within 200 m of land, i.e. `water ∩ buffer(land, 200 m)`

Islands count as shore, so the whole archipelago gets fine cells around every island. Fishing works the same way: you cast from island shores too.

In [ ]:
water = depth.union_all().intersection(domain_poly)
land = domain_poly.difference(water)
near = water.intersection(land.buffer(200))
print(f"water {water.area / 1e6:.0f} km², of which within 200 m of shore {near.area / 1e6:.0f} km²")

## 2c. Turn the polygons into H3 cells

`h3.geo_to_cells` returns every cell whose **centre** lies inside a polygon. It needs lon/lat, so we reproject first.

Mixing resolutions has one catch: a res 9 cell and a res 10 cell must not cover the same water. H3 is hierarchical, so every res 10 cell has exactly one res 9 parent. The rule:

1. Take all res 10 water cells (`water10`) and the res 10 cells in the nearshore band (`near10`).
2. Any res 9 parent with at least one nearshore child is **split**: all its water children stay as res 10.
3. Every other parent that holds water stays as **one res 9 cell**.

So each parent is either whole or fully split, with no overlaps and no gaps. The cost: the res 10 zone reaches a bit past 200 m (up to one res 9 cell, ~350 m across). Land is handled the same way. A res 9 cell that touches land always has a child within 200 m of it, so it gets split, and the children on land drop out.

In [ ]:
water_ll, near_ll = gpd.GeoSeries([water, near], crs=3067).to_crs(4326)
water10 = set(h3.geo_to_cells(water_ll, 10))
near10 = set(h3.geo_to_cells(near_ll, 10))

split_parents = {h3.cell_to_parent(c, 9) for c in near10}
res10 = {c for c in water10 if h3.cell_to_parent(c, 9) in split_parents}
res9 = {h3.cell_to_parent(c, 9) for c in water10} - split_parents

# self-check: no res 9 cell overlaps a res 10 cell, and every water cell is covered exactly once
assert not {h3.cell_to_parent(c, 9) for c in res10} & res9
assert water10 == res10 | {c for p in res9 for c in h3.cell_to_children(p, 10) if c in water10}

## 3. Cell count per resolution

In [ ]:
print(f"res 10 (nearshore): {len(res10):>6,}")
print(f"res  9 (offshore):  {len(res9):>6,}")
print(f"total:              {len(res10) + len(res9):>6,}")

## 4. Interactive map

Each cell becomes a hexagon polygon. Orange = res 10 nearshore, blue = res 9 offshore. Toggle the layers top right and hover a cell to see its H3 index.

~40k polygons make the map heavy, so zooming may lag a little. Clear this output before committing, or the notebook grows by several MB.

In [ ]:
cells = gpd.GeoDataFrame(
    {"h3": [*res10, *res9], "res": [10] * len(res10) + [9] * len(res9)},
    geometry=[shape(h3.cells_to_geo([c])) for c in [*res10, *res9]],
    crs=4326,
)
m = folium.Map(location=[60.13, 25.05], zoom_start=10)
for res, color, name in [(9, "#2b7bba", "res 9 offshore"), (10, "#e6550d", "res 10 nearshore")]:
    folium.GeoJson(
        cells[cells.res == res],
        name=name,
        style_function=lambda _, c=color: {"color": c, "weight": 0.3, "fillOpacity": 0.4},
        tooltip=folium.GeoJsonTooltip(["h3"]),
    ).add_to(m)
folium.GeoJson(domain.to_crs(4326), name="domain", style_function=lambda _: {"color": "black", "fill": False}).add_to(m)
folium.LayerControl().add_to(m)
m

## Notes for the go/no-go

- **Count vs spec:** compare the total above with the spec's "~30k". Most cells are nearshore, because the archipelago has a very long shoreline.
- **Chart gaps:** water with no depth area polygon (small inner bays, river mouths outside chart coverage) is treated as land here. Check the Vanhankaupunginlahti estuary on the map.
- **OSM shoreline:** the spec also names OSM for land removal. Here Traficom alone is used; whether OSM is needed is an open question for the spike.